# E15 extension — Rank invariance under the matched-alert-budget policy

**Extends:** the 2026-09-18 E15 findings entry in `DECISIONS.md`, which recorded that conformal
calibration changed no matched-budget alert. This notebook states the formal result behind that
finding, classifies every method/bound combination against it, and confirms the classification on
E15's own per-event scores. Those scores were not persisted, so the four learners and the GBM
quantile heads are refit with the **cached** hyperparameters: no search and no model selection. The
refit is deterministic, and §5 checks it against the saved E15 table. The official test set is
scored only.

## Setting
Events i = 1…n carry scores s_i. The matched-budget policy with budget K (E15, D3) alerts the K
highest-scoring events, and events tied at the boundary share the remaining alerts equally.
Formally, with s₍K₎ the K-th largest score:

a_i(K; s) = 1 if s_i > s₍K₎;  (K − #{j : s_j > s₍K₎}) / #{j : s_j = s₍K₎} if s_i = s₍K₎;  0 otherwise.

## Proposition 1 (rank invariance)
For scores s and s′ on the same events, **a(K; s′) = a(K; s) for every K ∈ {0,…,n} if and only if s
and s′ are order-isomorphic.** Order-isomorphic means that for all i, j: s_i < s_j ⇔ s′_i < s′_j, and
s_i = s_j ⇔ s′_i = s′_j. In particular, s′ = g(s) with g **strictly** increasing gives identical
alerts at every budget.

**Proof (⇐).** Order isomorphism preserves both the descending order and the tie classes. So the
event holding the K-th largest s′ lies in the same tie class as the one holding the K-th largest s.
The sets {s′ > s′₍K₎} and {s′ = s′₍K₎} therefore equal {s > s₍K₎} and {s = s₍K₎}, so a(K; s′) = a(K; s).
A strictly increasing g is injective and order-preserving, which is order isomorphism.

**Proof (⇒).** From the alert family, recover u_i = min{K : a_i(K) > 0} and v_i = min{K : a_i(K) = 1}.
Under the rule above, u_i = #{j : s_j > s_i} + 1 and v_i = #{j : s_j ≥ s_i}. So s_i = s_j ⇔ (u_i, v_i) =
(u_j, v_j), and s_i < s_j ⇔ v_j ≤ u_i − 1. The alert family therefore determines the weak order up
to isomorphism. If s and s′ give identical alerts at every K, their (u, v) coincide and the orders
are isomorphic. ∎

**Remark (strictness matters).** A *weakly* increasing g can merge distinct scores into a tie and
change the fractional alerts; `tests/test_rank_invariance.py` has a three-event example.

## Corollaries — the classification tested below
1. **Translations of the point prediction.** For every learner, the split-conformal bound p(x) + Q
   and the weighted-conformal bound p(x) + Q_w are translations of p(x), because Q is one number
   shared by all test events. This holds for both the one-sided upper bound (signed-residual
   score) and the upper edge of the two-sided interval (absolute-residual score). Such bounds raise
   exactly the point prediction's alerts at every budget.
   - For weighted conformal this depends on the implementation's **single representative test
     weight** (`weighted.py`, `rule_derived_weights`). A per-event test weight w(x) would make Q
     event-specific and void the corollary.
   - Unsupported events (bound = +∞) would also break it. There are none: 2,167/2,167 are supported.
2. **CQR — a translation of its quantile head, not of the point prediction.** The one-sided bound is
   q₁₋α(x) + Q. The two-sided upper edge is max(q_α/2(x), q₁₋α/2(x)) + Q, i.e. the crossing-repaired
   upper head plus Q. Both are rank-identical to their quantile head. The heads are separately fitted
   pinball-loss models, so there is no structural relation to p(x); any agreement with it is empirical.
3. **The Bayesian bound (E8) — event-specific dispersion.** It is μ(x) + z·σ(x), with σ² =
   σ²_epistemic(x) + σ²_aleatoric. It is rank-identical to μ only if the ordering by μ + zσ matches
   the ordering by μ on these events. That is guaranteed structurally only when σ is constant (or an
   order-preserving function of μ). Otherwise closeness is an empirical question: how small the
   variation in zσ is relative to the spacing of μ.
4. **Threshold rules (sets up the expanded E15/E16 design).** For the rule "alert iff s ≥ t" and a
   strictly increasing g: alerts(g(p), t) = alerts(p, g⁻¹(t)). A translation p + Q therefore traces
   **the same missed-vs-unnecessary operating locus** as p over all thresholds. Calibration moves
   only *which* operating point a given t selects: the one at t − Q.

In [ ]:
# --- Setup + provenance (invariant I4) ------------------------------------------------------
import json, subprocess, sys
from datetime import datetime, timezone

import numpy as np
import pandas as pd

from kelvins_conformal.config import REPO_ROOT, load_config
from kelvins_conformal.models import decision_runner as DR
from kelvins_conformal.reporting import write_table_atomic

cfg = load_config()
TABDIR = cfg.path("tables_dir"); TABDIR.mkdir(parents=True, exist_ok=True)

def git_sha():
    try:
        return subprocess.run(["git", "rev-parse", "HEAD"], cwd=str(REPO_ROOT),
                              capture_output=True, text=True, check=True).stdout.strip()
    except Exception:
        return "UNAVAILABLE"

def save_table(df, name):
    write_table_atomic(df, TABDIR / f"{name}.csv"); print(f"saved: reports/tables/{name}.csv")

PROVENANCE = {"experiment_ids": ["E15"], "analysis": "rank-invariance audit (Proposition 1)",
              "git_commit_sha": git_sha(), "config_hash": cfg.config_hash,
              "seeds": list(cfg.train.seeds),
              "executed_utc": datetime.now(timezone.utc).isoformat(), "python": sys.version.split()[0]}
print(json.dumps(PROVENANCE, indent=2))

## 1. Run the audit

In [ ]:
AUD = DR.run_rank_invariance_audit(cfg)
meta = AUD["meta"]
print(json.dumps(meta, indent=2, default=str))
for key in ("classification", "classification_raw", "alerts", "e8_dispersion", "budgets"):
    save_table(AUD[key], f"e15b_{key}")

## 2. Classification against Proposition 1

For each method, learner and sidedness, pooled over the three nominal levels and three seeds: is
the score order-isomorphic to the learner's point prediction, and to its construction reference?
`offset_spread_vs_point` is max − min of (score − point); it is 0 exactly for a translation.

In [ ]:
cls = AUD["classification"].copy()
TOL = 1e-9   # float headroom for a translation computed as point + Q

def verdict(r):
    if r.structural_class == DR.TRANSLATION_OF_POINT:
        ok = bool(r.strictly_increasing_in_point_all) and r.offset_spread_vs_point_max <= TOL
        return "CONFIRMED - rank-identical to its point prediction" if ok else "CONTRADICTED"
    if r.structural_class == DR.TRANSLATION_OF_QUANTILE_HEAD:
        if not bool(r.strictly_increasing_in_construction_reference_all):
            return "CONTRADICTED"
        return ("CONFIRMED - rank-identical to its quantile head, not to the point prediction"
                if not bool(r.strictly_increasing_in_point_all)
                else "CONFIRMED vs head; head happens to order events like the point (empirical)")
    return ("EMPIRICAL - rank-identical to its point prediction"
            if bool(r.strictly_increasing_in_point_all)
            else "EMPIRICAL - NOT rank-identical to its point prediction")

cls["verdict"] = [verdict(r) for r in cls.itertuples()]
view = cls[["method", "learner", "sided", "structural_class", "strictly_increasing_in_point_all",
            "order_violations_vs_point_max", "tie_violations_vs_point_max", "kendall_tau_vs_point_min",
            "offset_spread_vs_point_max", "strictly_increasing_in_construction_reference_all", "verdict"]]
display(view)
save_table(cls, "e15b_classification_verdicts")
N_CONTRADICTED = int(cls["verdict"].str.startswith("CONTRADICTED").sum())
print(f"Structural predictions contradicted by the data: {N_CONTRADICTED}")

## 3. Alerts at every pre-registered budget — identity and overlap with the point prediction

In [ ]:
al = AUD["alerts"]
prim = cfg.power.nominal_coverage_primary
v = al[al["nominal"] == prim]
display(v.pivot_table(index=["method", "learner", "sided"], columns="budget",
                      values="max_abs_alert_difference_vs_point").round(3))
display(v.pivot_table(index=["method", "learner", "sided"], columns="budget",
                      values="alert_overlap_vs_point_min").round(4))
tr = al[al["method"].isin([DR.E10, DR.E11, DR.E10_TWO, DR.E11_TWO])]
MAX_TRANSLATION_ALERT_DIFF = float(tr["max_abs_alert_difference_vs_point"].max())
print(f"Largest alert difference, translation class vs point, all levels/budgets/seeds: {MAX_TRANSLATION_ALERT_DIFF:.3g}")

## 4. The Bayesian bound: is its dispersion order-preserving in the mean?

Corollary 3 makes E8's agreement with its point prediction an empirical question. The quantities
that decide it: how much of the predictive variance is the constant aleatoric term; how widely z·σ
varies compared with the spread of μ; how σ co-moves with μ; and how many adjacent-pair order
violations remain.

In [ ]:
disp = AUD["e8_dispersion"]
display(disp.round(4))
e8 = cls[cls["structural_class"] == DR.EVENT_SPECIFIC_DISPERSION]
display(e8[["method", "sided", "strictly_increasing_in_point_all", "order_violations_vs_point_max",
            "tie_violations_vs_point_max", "kendall_tau_vs_point_min", "offset_spread_vs_point_max"]])
e8al = al[(al["method"].isin([DR.E8, DR.E8_TWO])) & (al["nominal"] == prim)]
display(e8al[["method", "budget", "K", "alert_overlap_vs_point_min", "missed_high_risk", "missed_high_risk_point"]].round(3))
E8_RANK_IDENTICAL = bool(e8["strictly_increasing_in_point_all"].all())
print(f"E8 bounds rank-identical to the MC mean on these events: {E8_RANK_IDENTICAL}")

## 5. Reproduction check against the saved E15 table

The refit is deterministic, so missed high-risk events for every E15 method must reproduce
`reports/tables/e15_decisions.csv`.

In [ ]:
e15 = pd.read_csv(TABDIR / "e15_decisions.csv")
up = al[al["sided"] == "upper"][["method", "learner", "nominal", "budget", "missed_high_risk"]]
pt = (al[al["method"] == DR.E10][["learner", "nominal", "budget", "missed_high_risk_point"]]
      .rename(columns={"missed_high_risk_point": "missed_high_risk"}).assign(method=DR.POINT))
mine = pd.concat([up, pt], ignore_index=True)
cmp = mine.merge(e15[["method", "learner", "nominal", "budget", "missed_high_risk"]],
                 on=["method", "learner", "nominal", "budget"], suffixes=("_audit", "_e15"))
cmp["abs_difference"] = (cmp["missed_high_risk_audit"] - cmp["missed_high_risk_e15"]).abs()
display(cmp.sort_values("abs_difference", ascending=False).head(10))
REPRO_MAX = float(cmp["abs_difference"].max())
print(f"Rows compared: {len(cmp)} of {len(e15)} E15 rows; largest |difference| in missed high-risk events: {REPRO_MAX:.3g}")

## 6. Findings — measurement only

In [ ]:
d = disp.mean(numeric_only=True)
print(f"""
RANK-INVARIANCE AUDIT - WHAT IT SHOWS (measurement only)

 Structural predictions contradicted: {N_CONTRADICTED} of {len(cls)} method/learner/sidedness combinations.
 Translation class (E10/E11, both sidednesses, all learners): largest alert difference vs point
 = {MAX_TRANSLATION_ALERT_DIFF:.3g}.
 E8 Bayesian bound rank-identical to MC mean: {E8_RANK_IDENTICAL}
   aleatoric share of mean predictive variance = {d['aleatoric_share_of_mean_variance']:.3f};
   range of z*sigma = {d['range_of_z_primary_times_std']:.3f} vs range of mu = {d['range_of_mean']:.3f};
   Spearman(mu, sigma) = {d['spearman_mean_vs_total_std']:.3f}.
 Reproduction of the E15 table: largest |difference| = {REPRO_MAX:.3g} over {len(cmp)} rows.

 NOT DECIDED HERE: confirmation of the proposition's use in the manuscript; anything in the
 expanded E15/E16 threshold analysis.
""")
(cfg.path("reports_dir") / "05b_rank_invariance_provenance.json").write_text(json.dumps(PROVENANCE, indent=2), encoding="utf-8")
print("provenance:", cfg.path("reports_dir") / "05b_rank_invariance_provenance.json")